# Chapter 7: Neural Networks

```{admonition} Learning Objectives
:class: tip
- Understand perceptron and multi-layer networks
- Implement backpropagation algorithm
- Apply different activation functions
- Build neural networks from scratch
- Use deep learning frameworks (PyTorch/TensorFlow)
- Prevent overfitting with dropout and batch normalization
```

## 7.1 From Linear Models to Neural Networks

Neural networks extend linear models with **non-linear transformations**.

### Single Neuron (Perceptron)

$$y = \sigma\left(\sum_{i=1}^d w_i x_i + b\right) = \sigma(\mathbf{w}^T\mathbf{x} + b)$$

where $\sigma$ is an **activation function**.

### Multi-Layer Perceptron (MLP)

$$\mathbf{h}^{(1)} = \sigma(\mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)})$$
$$\mathbf{h}^{(2)} = \sigma(\mathbf{W}^{(2)}\mathbf{h}^{(1)} + \mathbf{b}^{(2)})$$
$$\hat{\mathbf{y}} = \mathbf{W}^{(3)}\mathbf{h}^{(2)} + \mathbf{b}^{(3)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)

print("✓ Libraries imported!")

## 7.2 Activation Functions

Activation functions introduce **non-linearity** - crucial for learning complex patterns.

### Common Activations

1. **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}}$
   - Range: $(0, 1)$
   - Use: Output layer (binary classification)

2. **Tanh**: $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$
   - Range: $(-1, 1)$
   - Use: Hidden layers (zero-centered)

3. **ReLU**: $\text{ReLU}(z) = \max(0, z)$
   - Range: $[0, \infty)$
   - Use: Default for hidden layers

4. **Leaky ReLU**: $\text{LReLU}(z) = \max(\alpha z, z)$
   - Prevents "dying ReLU" problem

5. **Softmax**: $\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
   - Use: Multi-class output layer

In [ ]:
class Activations:
    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    @staticmethod
    def sigmoid_derivative(z):
        s = Activations.sigmoid(z)
        return s * (1 - s)
    
    @staticmethod
    def tanh(z):
        return np.tanh(z)
    
    @staticmethod
    def tanh_derivative(z):
        return 1 - np.tanh(z)**2
    
    @staticmethod
    def relu(z):
        return np.maximum(0, z)
    
    @staticmethod
    def relu_derivative(z):
        return (z > 0).astype(float)
    
    @staticmethod
    def leaky_relu(z, alpha=0.01):
        return np.where(z > 0, z, alpha * z)
    
    @staticmethod
    def leaky_relu_derivative(z, alpha=0.01):
        return np.where(z > 0, 1, alpha)
    
    @staticmethod
    def softmax(z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# Visualize activations
z = np.linspace(-5, 5, 100)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

activations = [
    ('Sigmoid', Activations.sigmoid),
    ('Tanh', Activations.tanh),
    ('ReLU', Activations.relu),
    ('Leaky ReLU', Activations.leaky_relu),
]

for ax, (name, func) in zip(axes.flat, activations):
    ax.plot(z, func(z), linewidth=2)
    ax.set_title(name)
    ax.grid(True)
    ax.axhline(0, color='k', linestyle='--', alpha=0.3)
    ax.axvline(0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Activation functions defined!")

## 7.3 Backpropagation

The **chain rule** applied to neural networks for computing gradients.

### Forward Pass

$$\mathbf{z}^{(l)} = \mathbf{W}^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$$
$$\mathbf{a}^{(l)} = \sigma(\mathbf{z}^{(l)})$$

### Backward Pass

**Output layer**:
$$\delta^{(L)} = \nabla_a L \odot \sigma'(\mathbf{z}^{(L)})$$

**Hidden layers**:
$$\delta^{(l)} = ((\mathbf{W}^{(l+1)})^T \delta^{(l+1)}) \odot \sigma'(\mathbf{z}^{(l)})$$

**Gradient**:
$$\frac{\partial L}{\partial \mathbf{W}^{(l)}} = \delta^{(l)} (\mathbf{a}^{(l-1)})^T$$
$$\frac{\partial L}{\partial \mathbf{b}^{(l)}} = \delta^{(l)}$$

In [ ]:
class NeuralNetwork:
    """Multi-layer Neural Network from scratch"""
    
    def __init__(self, layer_sizes, activation='relu', learning_rate=0.01):
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.activation = activation
        
        # Initialize weights (Xavier initialization)
        self.weights = []
        self.biases = []
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)
    
    def _activate(self, z):
        if self.activation == 'relu':
            return Activations.relu(z)
        elif self.activation == 'sigmoid':
            return Activations.sigmoid(z)
        elif self.activation == 'tanh':
            return Activations.tanh(z)
    
    def _activate_derivative(self, z):
        if self.activation == 'relu':
            return Activations.relu_derivative(z)
        elif self.activation == 'sigmoid':
            return Activations.sigmoid_derivative(z)
        elif self.activation == 'tanh':
            return Activations.tanh_derivative(z)
    
    def forward(self, X):
        """Forward propagation"""
        self.z_values = []
        self.activations = [X]
        
        for i in range(len(self.weights)):
            z = self.activations[-1] @ self.weights[i] + self.biases[i]
            self.z_values.append(z)
            
            if i == len(self.weights) - 1:
                # Output layer - sigmoid for binary classification
                a = Activations.sigmoid(z)
            else:
                # Hidden layers
                a = self._activate(z)
            
            self.activations.append(a)
        
        return self.activations[-1]
    
    def backward(self, X, y):
        """Backpropagation"""
        m = X.shape[0]
        
        # Output layer gradient
        delta = self.activations[-1] - y.reshape(-1, 1)
        
        # Store gradients
        weight_gradients = []
        bias_gradients = []
        
        # Backward through layers
        for i in range(len(self.weights) - 1, -1, -1):
            # Gradients
            dw = self.activations[i].T @ delta / m
            db = np.sum(delta, axis=0, keepdims=True) / m
            
            weight_gradients.insert(0, dw)
            bias_gradients.insert(0, db)
            
            # Propagate to previous layer
            if i > 0:
                delta = (delta @ self.weights[i].T) * self._activate_derivative(self.z_values[i-1])
        
        # Update weights
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * weight_gradients[i]
            self.biases[i] -= self.learning_rate * bias_gradients[i]
    
    def train(self, X, y, epochs=1000, verbose=False):
        losses = []
        
        for epoch in range(epochs):
            # Forward pass
            y_pred = self.forward(X)
            
            # Loss (binary cross-entropy)
            loss = -np.mean(y * np.log(y_pred + 1e-15) + (1-y) * np.log(1-y_pred + 1e-15))
            losses.append(loss)
            
            # Backward pass
            self.backward(X, y)
            
            if verbose and epoch % 100 == 0:
                acc = accuracy_score(y, (y_pred >= 0.5).astype(int))
                print(f"Epoch {epoch}: Loss = {loss:.4f}, Accuracy = {acc:.4f}")
        
        return losses
    
    def predict(self, X):
        y_pred = self.forward(X)
        return (y_pred >= 0.5).astype(int).flatten()
    
    def score(self, X, y):
        return accuracy_score(y, self.predict(X))

print("✓ Neural Network implemented!")

In [ ]:
# Example: Non-linear classification
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train neural network
nn = NeuralNetwork([2, 16, 8, 1], activation='relu', learning_rate=0.1)
losses = nn.train(X_train, y_train, epochs=1000, verbose=True)

# Evaluate
train_acc = nn.score(X_train, y_train)
test_acc = nn.score(X_test, y_test)
print(f"\nTrain Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Plot decision boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True)

# Decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200),
                    np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200))
Z = nn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
ax2.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdYlBu', edgecolors='k')
ax2.set_title(f'Neural Network Decision Boundary\nTest Accuracy: {test_acc:.3f}')
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 7.4 Modern Deep Learning Frameworks

While understanding the fundamentals is crucial, modern frameworks make implementation much easier.

In [ ]:
# Example with PyTorch (if available)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    
    class PyTorchNN(nn.Module):
        def __init__(self, input_size, hidden_sizes, output_size):
            super().__init__()
            layers = []
            prev_size = input_size
            
            for hidden_size in hidden_sizes:
                layers.append(nn.Linear(prev_size, hidden_size))
                layers.append(nn.ReLU())
                prev_size = hidden_size
            
            layers.append(nn.Linear(prev_size, output_size))
            layers.append(nn.Sigmoid())
            
            self.network = nn.Sequential(*layers)
        
        def forward(self, x):
            return self.network(x)
    
    # Convert to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
    
    # Create model
    model_pt = PyTorchNN(2, [16, 8], 1)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model_pt.parameters(), lr=0.01)
    
    # Train
    for epoch in range(500):
        optimizer.zero_grad()
        outputs = model_pt(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        
        if epoch % 100 == 0:
            print(f"PyTorch Epoch {epoch}: Loss = {loss.item():.4f}")
    
    print("✓ PyTorch implementation complete!")
    
except ImportError:
    print("PyTorch not installed. Install with: pip install torch")

## 7.5 Regularization Techniques

### Dropout

Randomly "drop" neurons during training to prevent co-adaptation.

$$\mathbf{h}^{(l)} = \mathbf{r} \odot \sigma(\mathbf{z}^{(l)})$$

where $r_i \sim \text{Bernoulli}(p)$.

### Batch Normalization

Normalize layer inputs to stabilize training:

$$\hat{z}_i = \frac{z_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$

### Early Stopping

Stop training when validation loss stops improving.

## 7.6 Summary

### Key Concepts

1. **Activation functions** enable non-linearity
2. **Backpropagation** = chain rule for gradients
3. **Universal approximation** - NNs can approximate any function
4. **Deep networks** learn hierarchical representations
5. **Regularization** prevents overfitting

### Design Choices

| Component | Options | Best Practice |
|-----------|---------|---------------|
| Hidden activation | ReLU, tanh, sigmoid | ReLU (default) |
| Output activation | Sigmoid, softmax, linear | Task-dependent |
| Initialization | Random, Xavier, He | Xavier/He |
| Optimizer | SGD, Adam, RMSprop | Adam (default) |
| Learning rate | 0.001-0.1 | 0.001 (Adam) |

## Exercises

```{exercise} XOR Problem
:label: ex-7-1

Train a 2-layer network to solve XOR. Visualize decision boundary and explain why 1 layer fails.
```

```{exercise} Activation Comparison
:label: ex-7-2

Compare ReLU, tanh, and sigmoid on the moons dataset. Which converges fastest? Why?
```

```{exercise} Vanishing Gradients
:label: ex-7-3

Build a 10-layer network with sigmoid activations. Observe gradient magnitudes. Compare with ReLU.
```

```{exercise} MNIST Digit Classification
:label: ex-7-4

Build a neural network to classify handwritten digits from MNIST dataset. Achieve >95% accuracy.
```

## Further Reading

- Aggarwal, C. C. (2021). *Artificial Intelligence*, Chapter 7
- Goodfellow et al. (2016). *Deep Learning*
- Nielsen, M. (2015). *Neural Networks and Deep Learning* (Online)
- 3Blue1Brown. *Neural Networks* (YouTube series)

---

**Next Chapter**: [Domain Architectures](ch08_domain_architectures.ipynb) - CNNs, RNNs, Transformers